# Custom Optimizers: SGD, Momentum, RMSprop, and Adam

This notebook provides a pure NumPy implementation of the four major optimization algorithms used in deep learning. We will implement their update formulas from scratch and trace their trajectories as they navigate a non-convex saddle-point function surface.

## 1. Mathematical Formulas

Let $w$ be the parameter vector and $dw$ represent the current gradient.

### Stochastic Gradient Descent (SGD)
$$w \leftarrow w - \alpha \cdot dw$$

### SGD with Momentum
- $v_d \leftarrow \beta_1 v_d + (1 - \beta_1) dw$
- $w \leftarrow w - \alpha \cdot v_d$

### RMSprop
- $s_d \leftarrow \beta_2 s_d + (1 - \beta_2) dw^2$
- $w \leftarrow w - \alpha \cdot \frac{dw}{\sqrt{s_d + \epsilon}}$

### Adam (Adaptive Moment Estimation)
- $v_d \leftarrow \beta_1 v_d + (1 - \beta_1) dw$ (First moment - velocity)
- $s_d \leftarrow \beta_2 s_d + (1 - \beta_2) dw^2$ (Second moment - variance)
- Bias Corrections for iteration $t$:
  - $v_d^{\text{corrected}} = \frac{v_d}{1 - \beta_1^t}$
  - $s_d^{\text{corrected}} = \frac{s_d}{1 - \beta_2^t}$
- Parameter Update:
  - $w \leftarrow w - \alpha \cdot \frac{v_d^{\text{corrected}}}{\sqrt{s_d^{\text{corrected}} + \epsilon}}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the saddle surface function: f(x, y) = x^2 - 1.5 * y^2
# Objective is to descend towards x=0, while y can oscillate
def cost_function(w):
    return w[0]**2 - 1.5 * w[1]**2

def compute_gradients(w):
    dw = np.zeros_like(w)
    dw[0] = 2.0 * w[0]               # Gradient wrt x
    dw[1] = -3.0 * w[1]              # Gradient wrt y
    # Add some random noise to simulate mini-batch variance
    dw += np.random.normal(0, 0.05, size=w.shape)
    return dw

## 2. Optimizer Implementations

In [ ]:
def optimize_sgd(w_start, lr, steps):
    w = np.array(w_start, dtype=float)
    path = [w.copy()]
    for _ in range(steps):
        dw = compute_gradients(w)
        w -= lr * dw
        path.append(w.copy())
    return np.array(path)

def optimize_momentum(w_start, lr, steps, beta=0.9):
    w = np.array(w_start, dtype=float)
    v = np.zeros_like(w)
    path = [w.copy()]
    for _ in range(steps):
        dw = compute_gradients(w)
        # Velocity update
        v = beta * v + (1 - beta) * dw
        w -= lr * v
        path.append(w.copy())
    return np.array(path)

def optimize_rmsprop(w_start, lr, steps, beta2=0.999, eps=1e-8):
    w = np.array(w_start, dtype=float)
    s = np.zeros_like(w)
    path = [w.copy()]
    for _ in range(steps):
        dw = compute_gradients(w)
        # Squared gradient moving average
        s = beta2 * s + (1 - beta2) * (dw**2)
        w -= lr * dw / (np.sqrt(s) + eps)
        path.append(w.copy())
    return np.array(path)

def optimize_adam(w_start, lr, steps, beta1=0.9, beta2=0.999, eps=1e-8):
    w = np.array(w_start, dtype=float)
    v = np.zeros_like(w)
    s = np.zeros_like(w)
    path = [w.copy()]
    for t in range(1, steps + 1):
        dw = compute_gradients(w)
        
        # Moment updates
        v = beta1 * v + (1 - beta1) * dw
        s = beta2 * s + (1 - beta2) * (dw**2)
        
        # Bias corrections
        v_corr = v / (1 - beta1**t)
        s_corr = s / (1 - beta2**t)
        
        w -= lr * v_corr / (np.sqrt(s_corr) + eps)
        path.append(w.copy())
    return np.array(path)

## 3. Visualize Optimizer Trajectories

In [ ]:
w_start = [2.0, 1.8]  # Starting position on the saddle point
steps = 30

path_sgd = optimize_sgd(w_start, lr=0.15, steps=steps)
path_momentum = optimize_momentum(w_start, lr=0.15, steps=steps)
path_rmsprop = optimize_rmsprop(w_start, lr=0.1, steps=steps)
path_adam = optimize_adam(w_start, lr=0.1, steps=steps)

plt.figure(figsize=(10, 8))

# Draw contours of the saddle surface
X, Y = np.meshgrid(np.linspace(-2.2, 2.2, 100), np.linspace(-2.2, 2.2, 100))
Z = X**2 - 1.5 * Y**2
plt.contour(X, Y, Z, levels=20, cmap='RdGy', alpha=0.3)

plt.plot(path_sgd[:, 0], path_sgd[:, 1], marker='o', label='SGD', color='#e03131', linewidth=1.5)
plt.plot(path_momentum[:, 0], path_momentum[:, 1], marker='s', label='Momentum', color='#f76707', linewidth=1.8)
plt.plot(path_rmsprop[:, 0], path_rmsprop[:, 1], marker='^', label='RMSprop', color='#339af0', linewidth=1.8)
plt.plot(path_adam[:, 0], path_adam[:, 1], marker='D', label='Adam', color='#2b8a3e', linewidth=2.2)

plt.title("Optimizer Performance Trajectories on Saddle Surface")
plt.xlabel("Parameter x (Descent direction)")
plt.ylabel("Parameter y (Oscillation direction)")
plt.xlim(-2.2, 2.2)
plt.ylim(-2.2, 2.2)
plt.legend(frameon=True, facecolor='#ffffff')
plt.grid(True, linestyle='--')
plt.show()

### Observations
1. **SGD** exhibits severe vertical oscillations. The step size scales with the gradient directly, bouncing wildly back and forth.
2. **Momentum** carries velocity from prior steps, smoothing out oscillations along the vertical y-axis.
3. **RMSprop** scales steps inversely with the running variance of gradients. This dampens vertical movements and accelerates horizontal descent.
4. **Adam** combines the best of both Momentum (dampening oscillations) and RMSprop (scaling steps), yielding the smoothest and fastest path to the minimum.